# FanDuel online casino / iGaming research (+ cross-product)

**Question:** Is FanDuel gaining or losing reported **online casino** revenue share, and does that trajectory differ from OSB handle share where both exist?

OSB handle analysis remains in notebook 92. Products are **not** summed. Formal approval remains **MA OSB only**.


## 0. Setup


In [ ]:
from __future__ import annotations

import hashlib
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
import sys

sys.path.insert(0, str(ROOT / "src"))
from variant_gaming.storage import connect_readonly

STAGING_DB = ROOT / "data/staging/gaming_nationwide.sqlite"
ORIGINAL_DB = ROOT / "data/gaming.sqlite"
EXPECTED = "023ca5e8e4c16ff0981a2783dabcedf9399939e0241701b0394bc0277eff6ce9"
C_BLUE, C_ORANGE, C_GRAY, C_BLACK = "#0072B2", "#E69F00", "#4D4D4D", "#000000"


def sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


h0, s0 = sha256(ORIGINAL_DB), sha256(STAGING_DB)
if s0 != EXPECTED:
    raise SystemExit("FAIL CLOSED: analyst-approval binding mismatch: staging snapshot changed; review required.")
print("staging", s0)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 100)


## 1. Online-casino FanDuel availability / gaps


In [ ]:
conn = connect_readonly(STAGING_DB)
raw = pd.read_sql_query(
    """
    SELECT state_code, channel, frequency, row_type, operator, period_start,
           handle, gross_revenue, adjusted_revenue, reported_revenue_name, report_status, source_sha256, source_file, source_url
    FROM gaming_results WHERE vertical='online_casino'
    """,
    conn,
)
conn.close()
raw["period_start"] = pd.to_datetime(raw["period_start"])

fan = raw[
    raw["row_type"].eq("operator")
    & raw["operator"].str.contains("FanDuel|FANDUEL|Fanduel", case=False, na=False)
]
gaps = []
for st, g in raw.groupby("state_code"):
    f = fan[fan.state_code.eq(st)]
    off = g[g.row_type.eq("official_statewide_total")]
    gaps.append(
        {
            "state": st,
            "channels": ",".join(sorted(g.channel.dropna().unique())),
            "frequency": ",".join(sorted(g.frequency.dropna().unique())),
            "has_official": bool(len(off)),
            "fanduel_native_names": sorted(f.operator.unique()) if len(f) else [],
            "fd_periods": int(f.period_start.nunique()) if len(f) else 0,
            "fd_min": f.period_start.min().date().isoformat() if len(f) else None,
            "fd_max": f.period_start.max().date().isoformat() if len(f) else None,
            "revenue_names": sorted(g.reported_revenue_name.dropna().unique())[:3],
            "decision": (
                "INCLUDE exploratory (MI Gross Receipts / gross_revenue storage)"
                if st == "MI" and len(f)
                else (
                    "EXCLUDE — FanDuel rows too sparse for Jan–Jul matched window"
                    if st == "NJ" and len(f)
                    else (
                        "EXCLUDE — no FanDuel native identity"
                        if not len(f)
                        else "EXCLUDE — insufficient comparable FanDuel coverage"
                    )
                )
            ),
        }
    )
gap_df = pd.DataFrame(gaps).sort_values("state")
display(Markdown("### Availability / gaps"))
display(gap_df)


## 2. MI online casino analysis (exploratory)

Selected metric: `gross_revenue`, native **Gross Receipts**, in USD for both FanDuel and the printed market total. `adjusted_revenue` is **Adjusted Gross Receipts** after source deductions; it remains a separate column and is not used here. Existing row-level labels can describe that other measure, so the analysis label follows the selected column and source definition.

No handle is retained for MI casino: hold cannot be separated. Market Gross Receipts growth includes all influences on market revenue; the market/share bridge is an accounting identity, not causal attribution. Compare the same seven months in both years using pooled dollars, not average monthly shares.


In [ ]:
mi = raw[raw.state_code.eq("MI") & raw.channel.eq("online") & raw.frequency.eq("monthly")].copy()
fd_name = "FanDuel (MotorCity Casino)"
PRIOR = [p.to_timestamp() for p in pd.period_range("2025-01", "2025-07", freq="M")]
CURRENT = [p.to_timestamp() for p in pd.period_range("2026-01", "2026-07", freq="M")]
WINDOW = PRIOR + CURRENT

# Keep every period's rejection reason and retained sources visible.
rows, checks = [], []
for ts, period in mi.groupby("period_start"):
    f = period[period.operator.eq(fd_name) & period.row_type.eq("operator")]
    o = period[period.row_type.eq("official_statewide_total")]
    ops = period[period.row_type.eq("operator")]
    reason, difference = "ok", float("nan")
    if len(f) != 1 or len(o) != 1 or ops.operator.duplicated().any():
        reason = "missing or duplicate identities/source versions; review required"
    elif period.gross_revenue.isna().any():
        reason = "missing Gross Receipts"
    elif float(o.gross_revenue.iloc[0]) <= 0:
        reason = "nonpositive market denominator"
    else:
        difference = float(ops.gross_revenue.sum()) - float(o.gross_revenue.iloc[0])
        if abs(difference) > 1.0:
            reason = "operator sum does not reconcile to printed total"
    checks.append({"month": ts, "validation": reason, "difference_usd": difference,
                   "missing_values": int(period.gross_revenue.isna().sum()),
                   "source_file": list(period.source_file.unique()),
                   "source_url": list(period.source_url.unique())})
    if reason != "ok":
        continue
    mkt, fd = float(o.gross_revenue.iloc[0]), float(f.gross_revenue.iloc[0])
    rows.append({"month": ts, "fd_rev": fd, "mkt_rev": mkt, "share": fd / mkt, "label": "Gross Receipts"})
coverage_checks = pd.DataFrame(checks, columns=["month", "validation", "difference_usd", "missing_values", "source_file", "source_url"]).set_index("month").reindex(WINDOW)
coverage_checks["validation"] = coverage_checks["validation"].fillna("no stored report")
display(coverage_checks)
if not coverage_checks.validation.eq("ok").all():
    raise SystemExit("FAIL CLOSED: MI Gross Receipts matched window incomplete; inspect coverage_checks.")
cas = pd.DataFrame(rows).sort_values("month")
display(cas[cas.month.isin(WINDOW)])
print("MI casino months available:", len(cas), cas.month.min().date(), "→", cas.month.max().date())


def pooled(months):
    s = cas[cas.month.isin(months)]
    return float(s.fd_rev.sum()), float(s.mkt_rev.sum()), float(s.fd_rev.sum() / s.mkt_rev.sum())


def symmetric_decomp(m0, s0, m1, s1):
    # revenue = market × share
    dm, ds = m1 - m0, s1 - s0
    mkt_effect = dm * (s0 + s1) / 2
    share_effect = ds * (m0 + m1) / 2
    delta = m1 * s1 - m0 * s0
    return mkt_effect, share_effect, delta


fd0, m0, share0 = pooled(PRIOR)
fd1, m1, share1 = pooled(CURRENT)
mkt_fx, sh_fx, delta = symmetric_decomp(m0, share0, m1, share1)
assert abs((mkt_fx + sh_fx) - delta) <= 0.01
print(
    f"Jan–Jul Gross Receipts share {100*share0:.2f}% → {100*share1:.2f}% ({100*(share1-share0):+.2f} pp); "
    f"FD rev YoY {100*(fd1/fd0-1):+.2f}%; market {100*(m1/m0-1):+.2f}%"
)
print(f"Accounting decomp: market-growth effect ${mkt_fx:,.0f}; share effect ${sh_fx:,.0f}; Δ ${delta:,.0f}")

# monthly share change heatmap (single state)
heat = []
for c, p in zip(CURRENT, PRIOR):
    a = cas[cas.month.eq(c)].iloc[0]
    b = cas[cas.month.eq(p)].iloc[0]
    heat.append({"month": c.strftime("%b"), "share_pp": 100 * (a.share - b.share)})
hdf = pd.DataFrame(heat).set_index("month").T
fig, ax = plt.subplots(figsize=(8.0, 1.6))
im = ax.imshow(hdf.to_numpy(), cmap="coolwarm", aspect="auto", vmin=-3, vmax=3)
ax.set_xticks(range(7), hdf.columns)
ax.set_yticks([0], ["MI"])
for j, v in enumerate(hdf.iloc[0]):
    ax.text(j, 0, f"{v:+.1f}", ha="center", va="center", fontsize=8)
ax.set_title("MI online casino: FanDuel Gross Receipts share change vs prior year (pp)")
fig.colorbar(im, ax=ax, fraction=0.05)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 2.2))
ax.barh(["MI"], [100 * (share1 - share0)], color=C_ORANGE if share1 > share0 else C_BLUE)
ax.axvline(0, color=C_BLACK, linewidth=0.8)
ax.set_xlabel("Gross Receipts share change (pp), Jan–Jul")
ax.set_title("MI casino ranked share change (single eligible state)")
fig.tight_layout()
plt.show()

# waterfall-ish decomp bars
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar([0, 1], [mkt_fx / 1e6, sh_fx / 1e6], color=[C_BLUE, C_ORANGE])
ax.set_xticks([0, 1], ["Market-growth\neffect", "Share\neffect"])
ax.set_ylabel("FanDuel Gross Receipts change ($m)")
ax.set_title("MI casino accounting decomposition (market × share)")
for i, v in enumerate([mkt_fx, sh_fx]):
    ax.text(i, v / 1e6 + (0.3 if v >= 0 else -0.8), f"${v/1e6:.1f}m", ha="center")
fig.tight_layout()
plt.show()

def share_direction(change):
    return "unchanged" if abs(change) < 1e-12 else ("gain" if change > 0 else "loss")

if abs(abs(mkt_fx) - abs(sh_fx)) <= 0.01:
    decomposition_note = "equal contribution magnitudes from market and share"
else:
    dominant = "Market" if abs(mkt_fx) > abs(sh_fx) else "Share"
    decomposition_note = f"{dominant} contribution has the larger absolute magnitude"
decomposition_note += f" (market ${mkt_fx:,.0f}; share ${sh_fx:,.0f}); accounting, not causal attribution."


## 3. Cross-product comparison (MI only — common coverage)


In [ ]:
# OSB MI from staging (handle share) — exploratory native identity
osb_connection = connect_readonly(STAGING_DB)
osb = pd.read_sql_query(
    """
    SELECT period_start, operator, row_type, handle, source_file, source_url
    FROM gaming_results
    WHERE state_code='MI' AND vertical='online_sports_betting' AND channel='online' AND frequency='monthly'
      AND period_start >= '2025-01-01' AND period_start <= '2026-07-01'
    """,
    osb_connection,
)
osb_connection.close()
osb["period_start"] = pd.to_datetime(osb["period_start"])


def osb_pooled(months):
    fd = osb[osb.period_start.isin(months) & osb.operator.eq(fd_name)]
    off = osb[osb.period_start.isin(months) & osb.row_type.eq("official_statewide_total")]
    ops = osb[osb.period_start.isin(months) & osb.row_type.eq("operator")]
    for frame in (fd, off):
        if frame.period_start.duplicated().any() or set(frame.period_start) != set(months) or frame.handle.isna().any():
            raise SystemExit("FAIL CLOSED: MI OSB comparison has missing or duplicate periods/values.")
    if ops.duplicated(["period_start", "operator"]).any() or ops.handle.isna().any():
        raise SystemExit("FAIL CLOSED: MI OSB operator values/identities require review.")
    differences = ops.groupby("period_start").handle.sum() - off.set_index("period_start").handle
    if differences.abs().gt(1).any() or off.handle.le(0).any():
        raise SystemExit("FAIL CLOSED: MI OSB denominator does not reconcile.")
    display(off[["period_start", "handle", "source_file", "source_url"]])
    return float(fd.handle.sum()), float(off.handle.sum())


ofd0, om0 = osb_pooled(PRIOR)
ofd1, om1 = osb_pooled(CURRENT)
cross = pd.DataFrame(
    [
        {
            "state": "MI",
            "OSB_handle_share_pp": 100 * (ofd1 / om1 - ofd0 / om0),
            "iGaming_revenue_share_pp": 100 * (share1 - share0),
            "OSB_rel_handle_growth_pp": 100 * (ofd1 / ofd0 - 1) - 100 * (om1 / om0 - 1),
            "iGaming_rel_revenue_growth_pp": 100 * (fd1 / fd0 - 1) - 100 * (m1 / m0 - 1),
            "caveats": "OSB=handle share; iGaming=Gross Receipts revenue share; measures not economically equivalent",
        }
    ]
)
display(Markdown("### Common-coverage direction table"))
display(cross)
cross_observation = (
    f"OSB: {share_direction(ofd1 / om1 - ofd0 / om0)} ({100*(ofd1/om1-ofd0/om0):+.2f} pp handle share); "
    f"casino: {share_direction(share1 - share0)} ({100*(share1-share0):+.2f} pp Gross Receipts share). "
    "Share-point changes are not equivalent economic impacts across these metrics."
)
print("MI Jan–Jul share direction — " + cross_observation)

print(
    "Hypothesis only: promotional intensity or product-mix differences — not identified from these tables."
)


## 4. Findings + integrity


In [ ]:
lines = [
    "**Casino coverage:** only MI currently supports a matched Jan–Jul FanDuel online-casino revenue-share analysis. NJ FanDuel casino rows are sparse; PA/CT/DE/WV/RI lack FanDuel native identity in staging.",
    f"**MI iGaming:** Jan–Jul Gross Receipts share {100*share0:.2f}% → {100*share1:.2f}% ({100*(share1-share0):+.2f} pp); {decomposition_note}",
    f"**Cross-product (MI):** {cross_observation}",
    (
        "**Missing data by question:** "
        "(1) **recovery durability** — post-July 2026 months; "
        "(2) **cross-product breadth** — brand-level FanDuel casino in NJ/PA with complete 2025–26 months; "
        "(3) **sports-mix attribution** — sport-level OSB handle; "
        "(4) **denominator confidence** — printed IN statewide OSB totals."
    ),
]
display(Markdown("### Findings"))
for ln in lines:
    display(Markdown("- " + ln))
assert sha256(ORIGINAL_DB) == h0 and sha256(STAGING_DB) == s0
print("DB hashes unchanged")
print(f"hash_staging={s0}")
print(f"hash_original={h0}")
